# Prepare immutable DIMSIM evaluation bundle

Edit the explicit paths and sampling parameters, then run all cells.


In [1]:
BUILD_METADATA = {'schema_version': 1, 'baseline_git_sha': '1f77df1526f79a9b67fd092adad79dbbf24b8d79', 'source_sha256': {'pinyin.py': '6d080de3613cd219edd5553512e17e4fe596a7bb881d59c0f6acd654bb85f1d3', 'evaluation/dimsim.py': '38712d92358e8378539935fd669f5e9a375193010007a3798faaea02c88e0195', 'evaluation/sampling.py': '39c473e643f1f30c62951d28d4e5b6004ed819256676ff351355d098ebeab0c8', 'evaluation/anchor_sampling.py': 'b1eed3343e60be7b068e4468df31827d892c66cd52a7a7842197506467dc52e8', 'evaluation/dimsim_sampling.py': 'a7a9784173322c27e046bd37c4d75c8e784307dc64a9c4c323ec4ca87b5713ca', 'evaluation/contextual_bundle.py': 'fe7b171ec1d54b35e87f6857da3425a8bfeaf55c368e7264679bc306eb852f6d', 'evaluation/bundle.py': 'aed6b3289523675685640a7f7a09688a81e0351a56b16416861791bd7ec72013'}, 'embedded_sha256': 'c9103dfe11d89fe454ef0c1a58085d9b6a999e70e60589ec0fc687569ed63bb3', 'generator_sha256': '57210dad6248f3a92d5198ea4a88363af339ef35b1a696d85135b13cc710bb98'}


In [2]:
import importlib.metadata
import subprocess
import sys

INSTALL_DEPENDENCIES = False
DEPENDENCIES = ["dimsim==0.2.2", "pypinyin==0.55.0", "numpy", "scipy", "pandas", "matplotlib"]
if INSTALL_DEPENDENCIES:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *DEPENDENCIES])

for package in ("dimsim", "pypinyin", "numpy", "scipy", "pandas", "matplotlib"):
    print(package, importlib.metadata.version(package))
if importlib.metadata.version("dimsim") != "0.2.2" or importlib.metadata.version("pypinyin") != "0.55.0":
    raise RuntimeError("Pinned DIMSIM preparation dependencies do not match")


dimsim 0.2.2
pypinyin 0.55.0
numpy 2.2.6
scipy 1.13.1
pandas 2.3.3
matplotlib 3.10.1


## Embedded `pinyin.py`


In [3]:
"""Project TONE3 input contract backed by pinned pypinyin inventories."""
from dataclasses import dataclass
from hashlib import sha256
from importlib.metadata import version
import re
from pypinyin.constants import PINYIN_DICT
from pypinyin.contrib.tone_convert import to_finals, to_initials, to_normal
PINYIN_CONTRACT_VERSION = 'pypinyin-tone3-1'

def _require_pypinyin_version():
    installed = version('pypinyin')
    if installed != '0.55.0':
        raise RuntimeError(f'pypinyin 0.55.0 required, found {installed}')
    return installed
_PYPINYIN_VERSION = _require_pypinyin_version()
VALID_SYLLABLES = frozenset((to_normal(reading, v_to_u=False).lower() for readings in PINYIN_DICT.values() for reading in readings.split(',')))
_INPUT_SYLLABLES = tuple(sorted((base for base in VALID_SYLLABLES if re.fullmatch('[a-z]+', base))))
_STRICT_COMPONENTS = tuple(((to_initials(base, strict=True), to_finals(base, strict=True, v_to_u=False)) for base in _INPUT_SYLLABLES))
INITIALS = sorted({initial for initial, final in _STRICT_COMPONENTS if final})
FINALS = sorted({final for _, final in _STRICT_COMPONENTS if final})
TONES = [str(tone) for tone in range(1, 6)]
PARSER_PROVENANCE = {'library': 'pypinyin', 'version': _PYPINYIN_VERSION, 'api': 'pypinyin.contrib.tone_convert', 'strict': True, 'input_format': 'lowercase-ascii-tone3', 'umlaut_storage': 'v', 'input_alias': 'u:', 'missing_tone': '5', 'empty_final': 'reject', 'inventory_source': 'pypinyin.constants.PINYIN_DICT', 'inventory_sha256': sha256('\n'.join(sorted(VALID_SYLLABLES)).encode('utf-8')).hexdigest(), 'vocabulary_sha256': sha256(repr((INITIALS, FINALS, TONES)).encode('utf-8')).hexdigest()}

@dataclass(frozen=True)
class PinyinComponents:
    initial: str
    final: str
    tone: str

def canonical_label(token: str, *, tone_convention='neutral_unmarked') -> str:
    """Accept one lowercase TONE3 syllable, allowing u: and unmarked neutral."""
    if tone_convention not in {'neutral_unmarked', 'explicit'}:
        raise ValueError(f'unsupported tone convention: {tone_convention!r}')
    if not isinstance(token, str) or not token:
        raise ValueError(f'Pinyin must be a non-empty string: {token!r}')
    if re.fullmatch('(?:[a-z]|u:)+[1-5]?', token) is None:
        raise ValueError(f'invalid Pinyin format (lowercase ASCII TONE3 required): {token!r}')
    canonical = token.replace('u:', 'v')
    if canonical[-1] not in TONES:
        if tone_convention == 'explicit':
            raise ValueError(f'explicit tone required: {token!r}')
        canonical += '5'
    if re.fullmatch('[a-z]+[1-5]', canonical) is None:
        raise ValueError(f'invalid canonical Pinyin format: {token!r}')
    base = canonical[:-1]
    if base not in VALID_SYLLABLES:
        raise ValueError(f'syllable absent from pypinyin dictionary: {token!r}')
    initial = to_initials(base, strict=True)
    final = to_finals(base, strict=True, v_to_u=False)
    if not final or initial not in INITIALS or final not in FINALS:
        raise ValueError(f'unsupported strict Pinyin components (non-empty final required): {token!r}')
    return canonical

def validate_canonical_label(token: str) -> str:
    """Reject noncanonical stored labels instead of silently repairing them."""
    if canonical_label(token, tone_convention='explicit') != token:
        raise ValueError(f'stored Pinyin label is not canonical: {token!r}')
    return token

def parse_components(token: str, *, tone_convention='neutral_unmarked') -> PinyinComponents:
    canonical = canonical_label(token, tone_convention=tone_convention)
    base = canonical[:-1]
    return PinyinComponents(to_initials(base, strict=True), to_finals(base, strict=True, v_to_u=False), canonical[-1])

def encode_components(components: PinyinComponents) -> tuple[int, int, int]:
    try:
        return (INITIALS.index(components.initial), FINALS.index(components.final), TONES.index(components.tone))
    except ValueError as exc:
        raise ValueError(f'components outside Pinyin vocabulary: {components!r}') from exc

def canonical_labels(tokens, *, tone_convention='neutral_unmarked'):
    return [canonical_label(token, tone_convention=tone_convention) for token in tokens]

def decompose(token: str, *, tone_convention='neutral_unmarked') -> tuple[int, int, int]:
    return encode_components(parse_components(token, tone_convention=tone_convention))


## Embedded `evaluation/dimsim.py`


In [4]:
"""Pinned public DIMSIM adapter used by evaluation-bundle preparation."""
import hashlib
from importlib import metadata
from pathlib import Path
DIMSIM_VERSION = '0.2.2'
PYPINYIN_VERSION = '0.55.0'

class DimsimDistance:
    """Validated callable around the public ``dimsim.get_distance`` API."""

    def __init__(self) -> None:
        versions = {'dimsim': metadata.version('dimsim'), 'pypinyin': metadata.version('pypinyin')}
        expected = {'dimsim': DIMSIM_VERSION, 'pypinyin': PYPINYIN_VERSION}
        if versions != expected:
            raise RuntimeError(f'DIMSIM dependency versions {versions} do not match pinned {expected}')
        import dimsim
        if not callable(getattr(dimsim, 'get_distance', None)):
            raise RuntimeError('dimsim.get_distance public API is unavailable')
        self._get_distance = dimsim.get_distance
        package_root = Path(dimsim.__file__).resolve().parent
        source_hashes = {}
        for path in sorted(package_root.rglob('*')):
            if path.is_file() and (path.suffix == '.py' or path.suffix.startswith('.pickle')):
                source_hashes[str(path.relative_to(package_root)).replace('\\', '/')] = hashlib.sha256(path.read_bytes()).hexdigest()
        self.provenance = {'metric_name': 'dimsim', 'metric_version': DIMSIM_VERSION, 'dependencies': versions, 'api': 'dimsim.get_distance([left], [right], pinyin=True)', 'canonical_weights_modified': False, 'package_file_sha256': source_hashes}

    def __call__(self, left: str, right: str) -> float:
        return float(self._get_distance([left], [right], pinyin=True))


## Embedded `evaluation/sampling.py`


In [5]:
"""Deterministic sampling of unique unordered pairs without quadratic materialization."""
import random
from collections.abc import Iterator, Sequence

def pair_count(item_count: int) -> int:
    """Return the number of unique unordered pairs of ``item_count`` objects."""
    if item_count < 0:
        raise ValueError('item_count must be non-negative')
    return item_count * (item_count - 1) // 2

def pair_from_rank(item_count: int, rank: int) -> tuple[int, int]:
    """Map a rank in lexicographic upper-triangle order to ``(left, right)``."""
    total = pair_count(item_count)
    if rank < 0 or rank >= total:
        raise IndexError(f'pair rank {rank} outside [0, {total})')
    low, high = (0, item_count - 1)
    while low + 1 < high:
        middle = (low + high) // 2
        before = middle * (2 * item_count - middle - 1) // 2
        if before <= rank:
            low = middle
        else:
            high = middle
    left = low
    before = left * (2 * item_count - left - 1) // 2
    right = left + 1 + (rank - before)
    return (left, right)

def iter_all_pairs(item_count: int, *, max_pairs: int) -> Iterator[tuple[int, int]]:
    """Yield all unique pairs, rejecting an unexpectedly large expansion."""
    total = pair_count(item_count)
    if max_pairs < 0:
        raise ValueError('max_pairs must be non-negative')
    if total > max_pairs:
        raise ValueError(f'all-pairs plan has {total} pairs, above cap {max_pairs}')
    for left in range(item_count):
        for right in range(left + 1, item_count):
            yield (left, right)

def sample_pair_indices(item_count: int, sample_count: int, seed: int) -> list[tuple[int, int]]:
    """Uniformly sample unique unordered pairs without constructing all pairs."""
    total = pair_count(item_count)
    if sample_count < 0:
        raise ValueError('sample_count must be non-negative')
    if sample_count > total:
        raise ValueError(f'requested {sample_count} pairs from population of {total}')
    ranks = random.Random(seed).sample(range(total), sample_count)
    return [pair_from_rank(item_count, rank) for rank in sorted(ranks)]

def make_pair_plan(item_ids: Sequence[str], *, sample_count: int | None, seed: int, max_all_pairs: int) -> list[tuple[str, str]]:
    """Return canonical ID pairs from an already stably ordered collection."""
    if len(item_ids) != len(set(item_ids)):
        raise ValueError('item_ids must be unique')
    if list(item_ids) != sorted(item_ids):
        raise ValueError('item_ids must be stably sorted')
    if sample_count is None:
        indices = iter_all_pairs(len(item_ids), max_pairs=max_all_pairs)
    else:
        indices = sample_pair_indices(len(item_ids), sample_count, seed)
    return [(item_ids[left], item_ids[right]) for left, right in indices]


## Embedded `evaluation/anchor_sampling.py`


In [6]:
"""Versioned hierarchical anchor-Q triplet sampling."""
import hashlib
import json
import numpy as np
SAMPLER_VERSION = 1
TRIPLET_SCHEMA_VERSION = 1
SEED = 20261026
EDGES = (('initial', '000', '100'), ('initial', '001', '101'), ('initial', '010', '110'), ('initial', '011', '111'), ('final', '000', '010'), ('final', '001', '011'), ('final', '100', '110'), ('final', '101', '111'), ('tone', '000', '001'), ('tone', '010', '011'), ('tone', '100', '101'), ('tone', '110', '111'))

def _stable_id(prefix: str, values: list[str]) -> str:
    payload = json.dumps(values, ensure_ascii=False, separators=(',', ':')).encode('utf-8')
    return prefix + hashlib.sha256(payload).hexdigest()

def sample_anchor_q(occurrences: list[dict], syllables: list[dict], *, draws_per_edge: int=1000, seed: int=SEED) -> tuple[list[dict], dict]:
    """Generate fixed draws; diagnostics are returned even for coverage failures."""
    if isinstance(draws_per_edge, bool) or not isinstance(draws_per_edge, int) or draws_per_edge <= 0:
        raise ValueError('draws_per_edge must be a positive integer')
    if isinstance(seed, bool) or not isinstance(seed, int) or seed < 0:
        raise ValueError('anchor-Q seed must be a non-negative integer')
    by_syllable = {row['syllable_id']: row for row in syllables}
    occurrence_map = {row['occurrence_id']: row for row in occurrences}
    members = {key: sorted(row['occurrence_ids']) for key, row in by_syllable.items()}

    def overlap(anchor_id: str, candidate_id: str) -> str:
        return ''.join((str(int(by_syllable[anchor_id][key] == by_syllable[candidate_id][key])) for key in ('initial', 'final', 'tone')))
    diagnostics = {'sampler_version': SAMPLER_VERSION, 'seed': seed, 'rng': 'numpy.random.PCG64', 'draws_per_edge': draws_per_edge, 'edges': {}}
    supports = {}
    candidate_supports = {}
    for component, low_mask, high_mask in EDGES:
        edge_id = f'{component}:{low_mask}->{high_mask}'
        eligible, excluded = ([], {})
        for sid in sorted(by_syllable):
            ids = members[sid]
            if not ids:
                excluded[sid] = 'no_occurrences'
                continue
            if high_mask == '111' and len(ids) < 2:
                excluded[sid] = 'high_111_requires_two_occurrences'
                continue
            high = [x for x in sorted(by_syllable) if overlap(sid, x) == high_mask and (x != sid or high_mask == '111')]
            low = [x for x in sorted(by_syllable) if overlap(sid, x) == low_mask and x != sid]
            if not high or not low:
                excluded[sid] = 'missing_high_or_low_syllable'
                continue
            eligible.append(sid)
            candidate_supports[component, low_mask, high_mask, sid] = (high, low)
        supports[edge_id] = (eligible, excluded)
        diagnostics['edges'][edge_id] = {'eligible_anchor_syllables': len(eligible), 'excluded_anchor_syllables': len(excluded), 'exclusion_reasons': excluded}
    missing = [edge for edge, (eligible, _) in supports.items() if not eligible]
    if missing:
        diagnostics['missing_edges'] = missing
        raise ValueError('anchor-Q coverage failure; diagnostics=' + json.dumps(diagnostics, ensure_ascii=False, sort_keys=True))
    rng = np.random.Generator(np.random.PCG64(seed))
    output = []
    for component, low_mask, high_mask in EDGES:
        edge_id = f'{component}:{low_mask}->{high_mask}'
        anchors = supports[edge_id][0]
        for draw_index in range(draws_per_edge):
            anchor_sid = anchors[int(rng.integers(len(anchors)))]
            anchor_occ = members[anchor_sid][int(rng.integers(len(members[anchor_sid])))]
            eligible_high, eligible_low = candidate_supports[component, low_mask, high_mask, anchor_sid]
            high_sid = eligible_high[int(rng.integers(len(eligible_high)))]
            low_sid = eligible_low[int(rng.integers(len(eligible_low)))]
            high_occ_choices = [x for x in members[high_sid] if x != anchor_occ]
            high_occ = high_occ_choices[int(rng.integers(len(high_occ_choices)))]
            low_occ_choices = members[low_sid]
            low_occ = low_occ_choices[int(rng.integers(len(low_occ_choices)))]
            if len({anchor_occ, high_occ, low_occ}) != 3:
                raise ValueError(f'anchor-Q sampler produced repeated occurrence IDs for {edge_id}')
            triple = [anchor_occ, high_occ, low_occ]
            triplet_id = _stable_id('triplet:', [edge_id, *triple])
            output.append({'schema_version': TRIPLET_SCHEMA_VERSION, 'sampler_version': SAMPLER_VERSION, 'edge_id': edge_id, 'component': component, 'low_mask': low_mask, 'high_mask': high_mask, 'draw_index': draw_index, 'draw_id': f'anchor-q:v1:{edge_id}:{draw_index:06d}', 'triplet_id': triplet_id, 'anchor_syllable_id': anchor_sid, 'high_syllable_id': high_sid, 'low_syllable_id': low_sid, 'anchor_id': anchor_occ, 'high_id': high_occ, 'low_id': low_occ})
    diagnostics['draw_count'] = len(output)
    diagnostics['unique_triplets_by_edge'] = {edge: len({r['triplet_id'] for r in output if r['edge_id'] == edge}) for edge in supports}
    diagnostics['unique_anchor_occurrences_by_edge'] = {edge: len({r['anchor_id'] for r in output if r['edge_id'] == edge}) for edge in supports}
    diagnostics['unique_anchor_syllables_by_edge'] = {edge: len({r['anchor_syllable_id'] for r in output if r['edge_id'] == edge}) for edge in supports}
    diagnostics['unique_occurrences_by_edge'] = {edge: len({r[k] for r in output if r['edge_id'] == edge for k in ('anchor_id', 'high_id', 'low_id')}) for edge in supports}
    for edge in supports:
        diagnostics['edges'][edge].update({'draw_count': sum((row['edge_id'] == edge for row in output)), 'unique_triplets': diagnostics['unique_triplets_by_edge'][edge], 'unique_anchor_occurrences': diagnostics['unique_anchor_occurrences_by_edge'][edge], 'unique_anchor_syllables': diagnostics['unique_anchor_syllables_by_edge'][edge], 'unique_occurrences': diagnostics['unique_occurrences_by_edge'][edge]})
    return (output, diagnostics)


## Embedded `evaluation/dimsim_sampling.py`


In [7]:
"""Occurrence-only DIMSIM sampling plans for schema-3 bundles."""
import hashlib
import json
import numpy as np
SAMPLER_VERSION = 1
PAIR_SCHEMA_VERSION = 1

def pair_identity(left: str, right: str) -> str:
    endpoints = sorted((left, right))
    raw = json.dumps(endpoints, ensure_ascii=False, separators=(',', ':')).encode('utf-8')
    return 'pair:' + hashlib.sha256(raw).hexdigest()

def sample_natural(occurrences: list[dict], count: int=12000, seed: int=42) -> list[tuple[str, str, str, int]]:
    ids = sorted((row['occurrence_id'] for row in occurrences if row['initial']))
    population = len(ids) * (len(ids) - 1) // 2
    if isinstance(count, bool) or not isinstance(count, int) or count < 0 or (count > population):
        raise ValueError(f'natural DIMSIM sample_count {count} exceeds eligible unique pair population {population}')
    selected = sample_pair_indices(len(ids), count, seed)
    return [(ids[i], ids[j], pair_identity(ids[i], ids[j]), index) for index, (i, j) in enumerate(selected)]

def sample_balanced(occurrences: list[dict], syllables: list[dict], count: int=12000, seed: int=20261026) -> list[tuple[str, str, str, int, str, str]]:
    eligible = {row['syllable_id']: sorted(row['occurrence_ids']) for row in syllables if row['initial']}
    ids = sorted(eligible)
    if not ids:
        raise ValueError('balanced DIMSIM has no nonzero-initial syllables')
    rng = np.random.Generator(np.random.PCG64(seed))
    result = []
    for index in range(count):
        first_sid = ids[int(rng.integers(len(ids)))]
        seconds = [sid for sid in ids if sid != first_sid or len(eligible[sid]) >= 2]
        if not seconds:
            raise ValueError('balanced DIMSIM coverage failure: no eligible second syllable')
        second_sid = seconds[int(rng.integers(len(seconds)))]
        first_id = eligible[first_sid][int(rng.integers(len(eligible[first_sid])))]
        second_choices = [x for x in eligible[second_sid] if x != first_id]
        second_id = second_choices[int(rng.integers(len(second_choices)))]
        result.append((first_id, second_id, pair_identity(first_id, second_id), index, first_sid, second_sid))
    return result


## Embedded `evaluation/contextual_bundle.py`


In [8]:
"""Schema-3 immutable contextual-only evaluation bundle implementation."""
import hashlib
import importlib.metadata
import json
import math
import shutil
from pathlib import Path
from typing import Any
SCHEMA_VERSION = 3
Q_SEED = 20261026
Q_SAMPLER_VERSION = 1
TRIPLET_SCHEMA_VERSION = 1
DIMSIM_SAMPLER_VERSION = 1
PAIR_SCHEMA_VERSION = 1
FILES = {'config.json', 'manifest.json', 'coverage.json', 'sentences.jsonl', 'occurrences.jsonl', 'syllables.jsonl', 'pairs/dimsim-natural.jsonl', 'pairs/dimsim-balanced.jsonl', 'triplets/anchor-q.jsonl'}
SOURCE_HASH_NAMES = {'manifest.json', 'config.json', 'vocab.json', 'samples.jsonl', 'splits.json'}

def _sha(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()

def _json(path: Path, value: Any) -> None:
    path.write_text(json.dumps(value, ensure_ascii=False, sort_keys=True, indent=2) + '\n', encoding='utf-8')

def _jsonl(path: Path, rows: list[dict]) -> None:
    path.write_text(''.join((json.dumps(row, ensure_ascii=False, sort_keys=True) + '\n' for row in rows)), encoding='utf-8')

def _read(path: Path) -> Any:
    return json.loads(path.read_text(encoding='utf-8'))

def _readl(path: Path) -> list[dict]:
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line]

def _stable(prefix: str, values: list[str]) -> str:
    return prefix + hashlib.sha256(json.dumps(values, ensure_ascii=False, separators=(',', ':')).encode('utf-8')).hexdigest()

def _syllables(occurrences: list[dict]) -> list[dict]:
    members: dict[str, list[str]] = {}
    metadata = {}
    for row in occurrences:
        sid = _stable('syllable:', [row['pinyin']])
        if row.get('syllable_id') != sid:
            raise ValueError(f"occurrence syllable_id disagrees with canonical Pinyin: {row.get('occurrence_id')}")
        metadata[sid] = {'syllable_id': sid, 'pinyin': row['pinyin'], 'initial': row['initial'], 'final': row['final'], 'tone': row['tone']}
        members.setdefault(sid, []).append(row['occurrence_id'])
    return [{**metadata[sid], 'occurrence_ids': sorted(members[sid]), 'occurrence_count': len(members[sid])} for sid in sorted(metadata)]

def _anchor_q_support(syllables: list[dict]) -> dict[str, tuple[list[str], dict[str, str]]]:
    """Independently derive eligible anchors and exclusions from gold metadata."""
    by_id = {row['syllable_id']: row for row in syllables}
    ordered = sorted(by_id)

    def overlap(anchor_id: str, candidate_id: str) -> str:
        a, b = (by_id[anchor_id], by_id[candidate_id])
        return ''.join((str(int(a[key] == b[key])) for key in ('initial', 'final', 'tone')))
    support = {}
    for component, low_mask, high_mask in EDGES:
        eligible, excluded = ([], {})
        for anchor_id in ordered:
            members = by_id[anchor_id]['occurrence_ids']
            if high_mask == '111' and len(members) < 2:
                excluded[anchor_id] = 'high_111_requires_two_occurrences'
                continue
            high = [candidate for candidate in ordered if overlap(anchor_id, candidate) == high_mask and (candidate != anchor_id or high_mask == '111')]
            low = [candidate for candidate in ordered if overlap(anchor_id, candidate) == low_mask and candidate != anchor_id]
            if high and low:
                eligible.append(anchor_id)
            else:
                excluded[anchor_id] = 'missing_high_or_low_syllable'
        support[f'{component}:{low_mask}->{high_mask}'] = (eligible, excluded)
    return support

def _anchor_q_coverage(syllables: list[dict], triplets: list[dict], *, seed: int, draws_per_edge: int) -> dict:
    support = _anchor_q_support(syllables)
    by_edge = {edge: [row for row in triplets if row['edge_id'] == edge] for edge in support}
    result = {'sampler_version': Q_SAMPLER_VERSION, 'seed': seed, 'rng': 'numpy.random.PCG64', 'draws_per_edge': draws_per_edge, 'edges': {}}
    unique_triples, unique_anchor_occurrences, unique_anchor_syllables, unique_occurrences = ({}, {}, {}, {})
    for edge, (eligible, excluded) in support.items():
        rows = by_edge[edge]
        unique_triples[edge] = len({row['triplet_id'] for row in rows})
        unique_anchor_occurrences[edge] = len({row['anchor_id'] for row in rows})
        unique_anchor_syllables[edge] = len({row['anchor_syllable_id'] for row in rows})
        unique_occurrences[edge] = len({row[field] for row in rows for field in ('anchor_id', 'high_id', 'low_id')})
        result['edges'][edge] = {'eligible_anchor_syllables': len(eligible), 'excluded_anchor_syllables': len(excluded), 'exclusion_reasons': excluded, 'draw_count': len(rows), 'unique_triplets': unique_triples[edge], 'unique_anchor_occurrences': unique_anchor_occurrences[edge], 'unique_anchor_syllables': unique_anchor_syllables[edge], 'unique_occurrences': unique_occurrences[edge]}
    result.update({'draw_count': len(triplets), 'unique_triplets_by_edge': unique_triples, 'unique_anchor_occurrences_by_edge': unique_anchor_occurrences, 'unique_anchor_syllables_by_edge': unique_anchor_syllables, 'unique_occurrences_by_edge': unique_occurrences})
    return result

def _metric_rows(plans, occurrence_map, scorer, metric_name, mode):
    rows = []
    for item in plans:
        left_id, right_id, pair_id, draw_index = item[:4]
        left, right = (occurrence_map[left_id], occurrence_map[right_id])
        status, value = ('not_computed', None)
        if scorer is not None:
            try:
                value = float(scorer(left['pinyin'], right['pinyin']))
                if not math.isfinite(value) or value < 0:
                    raise ValueError('DIMSIM distance must be finite and non-negative')
                status = 'ok'
            except Exception as error:
                status, value = (f'invalid:{type(error).__name__}', None)
        mask = ''.join((str(int(left[key] == right[key])) for key in ('initial', 'final', 'tone')))
        row = {'schema_version': PAIR_SCHEMA_VERSION, 'sampler_version': DIMSIM_SAMPLER_VERSION, 'mode': mode, 'pair_id': pair_id, 'left_id': min(left_id, right_id), 'right_id': max(left_id, right_id), 'left_pinyin': occurrence_map[min(left_id, right_id)]['pinyin'], 'right_pinyin': occurrence_map[max(left_id, right_id)]['pinyin'], 'dimsim_distance': value, 'distance_status': status, 'overlap_mask': mask, 'equal_initial': mask[0] == '1', 'equal_final': mask[1] == '1', 'equal_tone': mask[2] == '1', 'left_zero_initial': False, 'right_zero_initial': False, 'no_zero_initial_diagnostic': True, 'draw_index': draw_index}
        if mode == 'balanced':
            row.update({'draw_id': f'dimsim-balanced:v1:{draw_index:06d}', 'first_id': left_id, 'second_id': right_id, 'first_syllable_id': item[4], 'second_syllable_id': item[5]})
        else:
            row['draw_id'] = f'dimsim-natural:v1:{draw_index:06d}'
        rows.append(row)
    return rows

def _validate_provenance(code_provenance):
    if not isinstance(code_provenance, dict) or not code_provenance or any((not isinstance(k, str) or not isinstance(v, str) or len(v) != 64 or any((c not in '0123456789abcdef' for c in v)) for k, v in code_provenance.items())):
        raise ValueError('explicit preparation code_provenance hashes are required')

def _validate_source_provenance(source_experiment):
    if not isinstance(source_experiment, dict) or set(source_experiment) != {'path', 'experiment_id', 'hashes'}:
        raise ValueError('source_experiment provenance must contain path, experiment_id, and hashes')
    for field in ('path', 'experiment_id'):
        value = source_experiment.get(field)
        if not isinstance(value, str) or not value.strip():
            raise ValueError(f'source_experiment {field} must be a non-empty string')
    hashes = source_experiment.get('hashes')
    if not isinstance(hashes, dict) or set(hashes) != SOURCE_HASH_NAMES or any((not isinstance(value, str) or len(value) != 64 or any((ch not in '0123456789abcdef' for ch in value)) for value in hashes.values())):
        raise ValueError('source_experiment hashes must contain exactly five valid SHA-256 values')

def _typed_equal(actual, expected) -> bool:
    """Compare JSON contract values without Python bool/int equality aliases."""
    if type(actual) is not type(expected):
        return False
    if isinstance(expected, dict):
        return set(actual) == set(expected) and all((_typed_equal(actual[key], value) for key, value in expected.items()))
    if isinstance(expected, list):
        return len(actual) == len(expected) and all((_typed_equal(left, right) for left, right in zip(actual, expected)))
    return actual == expected

def prepare_contextual_bundle(evaluation_id: str, *, experiment_path, output_root, split='test', sentence_limit=None, sentence_seed=42, distance_fn=None, metric_provenance=None, code_provenance=None, protocol_config=None) -> Path:
    evaluation_id = _validate_evaluation_id(evaluation_id)
    _validate_provenance(code_provenance)
    if not isinstance(protocol_config, dict):
        raise TypeError('protocol_config must be a dictionary')
    defaults = {'dimsim_natural_count': 12000, 'dimsim_natural_seed': 42, 'dimsim_balanced_count': 12000, 'dimsim_balanced_seed': 20261026, 'anchor_q_draws_per_edge': 1000, 'anchor_q_seed': Q_SEED}
    unknown = set(protocol_config) - set(defaults)
    if unknown:
        raise ValueError(f'unknown protocol_config keys: {sorted(unknown)}')
    settings = {**defaults, **protocol_config}
    for key, value in settings.items():
        if isinstance(value, bool) or not isinstance(value, int) or value < 0:
            raise ValueError(f'{key} must be a non-negative integer')
    for count_key in ('dimsim_natural_count', 'dimsim_balanced_count', 'anchor_q_draws_per_edge'):
        if settings[count_key] == 0:
            raise ValueError(f'{count_key} must be positive')
    if distance_fn is None or not callable(distance_fn):
        raise ValueError('an explicit distance provider is required')
    if not isinstance(metric_provenance, dict) or not isinstance(metric_provenance.get('metric_name'), str):
        raise ValueError('metric_provenance with metric_name is required')
    if metric_provenance['metric_name'] == 'dimsim':
        if not isinstance(distance_fn, DimsimDistance) or metric_provenance != distance_fn.provenance:
            raise ValueError('DIMSIM bundles require the pinned DimsimDistance provider and provenance')
    elif metric_provenance['metric_name'] != 'synthetic':
        raise ValueError('only DIMSIM and explicitly synthetic metric providers are supported')
    source, root = (Path(experiment_path).resolve(), Path(output_root).resolve())
    target = (root / evaluation_id).resolve()
    if target.parent != root or target == source or source in target.parents:
        raise ValueError('invalid evaluation output path')
    if target.exists():
        raise FileExistsError(f'evaluation bundle already exists: {target}')
    samples, vocab, source_manifest, source_hashes = _load_source_experiment(source, split, sentence_limit, sentence_seed)
    occurrences = _build_occurrences(samples, vocab)
    for row in occurrences:
        row['syllable_id'] = _stable('syllable:', [row['pinyin']])
    syllables = _syllables(occurrences)
    triplets, q_coverage = sample_anchor_q(occurrences, syllables, draws_per_edge=settings['anchor_q_draws_per_edge'], seed=settings['anchor_q_seed'])
    plans_natural = sample_natural(occurrences, settings['dimsim_natural_count'], settings['dimsim_natural_seed'])
    plans_balanced = sample_balanced(occurrences, syllables, settings['dimsim_balanced_count'], settings['dimsim_balanced_seed'])
    if sample_anchor_q(occurrences, syllables, draws_per_edge=settings['anchor_q_draws_per_edge'], seed=settings['anchor_q_seed'])[0] != triplets:
        raise RuntimeError('anchor-Q deterministic preparation replay mismatch')
    if sample_natural(occurrences, settings['dimsim_natural_count'], settings['dimsim_natural_seed']) != plans_natural:
        raise RuntimeError('natural DIMSIM deterministic preparation replay mismatch')
    if sample_balanced(occurrences, syllables, settings['dimsim_balanced_count'], settings['dimsim_balanced_seed']) != plans_balanced:
        raise RuntimeError('balanced DIMSIM deterministic preparation replay mismatch')
    occurrence_map = {r['occurrence_id']: r for r in occurrences}
    natural = _metric_rows(plans_natural, occurrence_map, distance_fn, metric_provenance['metric_name'], 'natural')
    balanced = _metric_rows(plans_balanced, occurrence_map, distance_fn, metric_provenance['metric_name'], 'balanced')
    selected_sentences = [{'sentence_id': r['sample_id'], 'text': r['text']} for r in sorted(samples, key=lambda x: x['sample_id'])]
    coverage = {'sentence_count': len(selected_sentences), 'occurrence_count': len(occurrences), 'syllable_count': len(syllables), 'zero_initial_occurrences_excluded_from_dimsim': sum((not r['initial'] for r in occurrences)), 'dimsim_natural': {'draw_count': len(natural), 'unique_pairs': len({r['pair_id'] for r in natural}), 'unique_occurrences': len({r[k] for r in natural for k in ('left_id', 'right_id')}), 'unique_syllables': len({sid for r in natural for sid in (_stable('syllable:', [r['left_pinyin']]), _stable('syllable:', [r['right_pinyin']]))})}, 'dimsim_balanced': {'draw_count': len(balanced), 'unique_pairs': len({r['pair_id'] for r in balanced}), 'unique_occurrences': len({r[k] for r in balanced for k in ('left_id', 'right_id')}), 'unique_syllables': len({sid for r in balanced for sid in (r['first_syllable_id'], r['second_syllable_id'])}), 'same_pronunciation_draws': sum((r['overlap_mask'] == '111' for r in balanced))}, 'anchor_q': q_coverage}
    package_versions = {}
    for package in ('numpy', 'dimsim', 'pypinyin'):
        try:
            package_versions[package] = importlib.metadata.version(package)
        except importlib.metadata.PackageNotFoundError:
            package_versions[package] = None
    config = {'schema_version': SCHEMA_VERSION, 'evaluation_id': evaluation_id, 'split': split, 'sentence_limit': sentence_limit, 'sentence_seed': sentence_seed, 'population': {'unit': 'contextual_occurrence', 'zero_initial_excluded_from_dimsim': True, 'syllable_identity': 'canonical_toned_pinyin', 'aggregates': False}, 'sampling': {'dimsim_natural': {'count': settings['dimsim_natural_count'], 'without_replacement': True, 'rng': 'random.Random', 'seed': settings['dimsim_natural_seed'], 'sampler_version': DIMSIM_SAMPLER_VERSION, 'pair_schema_version': PAIR_SCHEMA_VERSION}, 'dimsim_balanced': {'count': settings['dimsim_balanced_count'], 'with_replacement': True, 'rng': 'numpy.random.PCG64', 'seed': settings['dimsim_balanced_seed'], 'choice_order': ['first_syllable', 'second_syllable', 'first_occurrence', 'second_occurrence'], 'sampler_version': DIMSIM_SAMPLER_VERSION, 'pair_schema_version': PAIR_SCHEMA_VERSION}, 'anchor_q': {'draws_per_edge': settings['anchor_q_draws_per_edge'], 'edge_order': [f'{c}:{l}->{h}' for c, l, h in EDGES], 'with_replacement': True, 'rng': 'numpy.random.PCG64', 'seed': settings['anchor_q_seed'], 'sampler_version': Q_SAMPLER_VERSION, 'triplet_schema_version': TRIPLET_SCHEMA_VERSION}}, 'scoring': {'dimsim': metric_provenance, 'anchor_q': {'cosine': 'stable_float64_scale_before_norm', 'tie_tolerance': 1e-12}}, 'runtime_versions': package_versions}
    source_provenance = {'path': str(source), 'experiment_id': source_manifest.get('experiment_id'), 'hashes': source_hashes}
    _validate_source_provenance(source_provenance)
    target.mkdir(parents=True)
    try:
        (target / 'pairs').mkdir()
        (target / 'triplets').mkdir()
        _jsonl(target / 'sentences.jsonl', selected_sentences)
        _jsonl(target / 'occurrences.jsonl', occurrences)
        _jsonl(target / 'syllables.jsonl', syllables)
        _jsonl(target / 'pairs/dimsim-natural.jsonl', natural)
        _jsonl(target / 'pairs/dimsim-balanced.jsonl', balanced)
        _jsonl(target / 'triplets/anchor-q.jsonl', triplets)
        _json(target / 'coverage.json', coverage)
        _json(target / 'config.json', config)
        artifacts = {p.relative_to(target).as_posix(): _sha(p) for p in target.rglob('*') if p.is_file()}
        manifest = {'schema_version': SCHEMA_VERSION, 'evaluation_id': evaluation_id, 'pinyin_contract': PINYIN_CONTRACT_VERSION, 'parser_provenance': PARSER_PROVENANCE, 'source_experiment': source_provenance, 'metric': metric_provenance, 'code_provenance': code_provenance, 'counts': {'sentences': len(selected_sentences), 'occurrences': len(occurrences), 'syllables': len(syllables), 'dimsim_natural': len(natural), 'dimsim_balanced': len(balanced), 'anchor_q': len(triplets)}, 'artifacts': artifacts}
        _json(target / 'manifest.json', manifest)
    except Exception:
        shutil.rmtree(target, ignore_errors=True)
        raise
    return target

def load_contextual_bundle(root: Path, manifest: dict) -> dict[str, Any]:
    if type(manifest.get('schema_version')) is not int or manifest.get('schema_version') != SCHEMA_VERSION:
        raise ValueError('unsupported evaluation bundle schema')
    if manifest.get('pinyin_contract') != PINYIN_CONTRACT_VERSION or manifest.get('parser_provenance') != PARSER_PROVENANCE:
        raise ValueError('unsupported evaluation Pinyin contract')
    _validate_provenance(manifest.get('code_provenance'))
    _validate_source_provenance(manifest.get('source_experiment'))
    config = _read(root / 'config.json')
    if config.get('schema_version') != SCHEMA_VERSION:
        raise ValueError('unsupported evaluation config schema')
    if _validate_id(config.get('evaluation_id')) != _validate_id(manifest.get('evaluation_id')):
        raise ValueError('schema-3 evaluation IDs disagree')
    _validate_config(config, manifest.get('metric'))
    artifacts = manifest.get('artifacts')
    if not isinstance(artifacts, dict) or set(artifacts) != FILES - {'manifest.json'}:
        raise ValueError('schema-3 manifest artifact inventory is invalid')
    actual = {p.relative_to(root).as_posix() for p in root.rglob('*') if p.is_file() and p.name != 'manifest.json'}
    if actual != FILES - {'manifest.json'}:
        raise ValueError('schema-3 bundle file inventory is invalid')
    for rel, expected in artifacts.items():
        path = (root / rel).resolve()
        if root not in path.parents or not path.is_file() or _sha(path) != expected:
            raise ValueError(f'evaluation artifact checksum mismatch: {rel}')
    sentences, occurrences, syllables = (_readl(root / name) for name in ('sentences.jsonl', 'occurrences.jsonl', 'syllables.jsonl'))
    sentence_map = {r.get('sentence_id'): r for r in sentences}
    occurrence_map = {r.get('occurrence_id'): r for r in occurrences}
    syllable_map = {r.get('syllable_id'): r for r in syllables}
    if len(sentence_map) != len(sentences) or len(occurrence_map) != len(occurrences) or len(syllable_map) != len(syllables):
        raise ValueError('duplicate or invalid sentence, occurrence, or syllable IDs')
    positions_by_sentence: dict[str, list[int]] = {sid: [] for sid in sentence_map}
    expected_syllables = _syllables(occurrences)
    if expected_syllables != syllables:
        raise ValueError('syllable metadata or membership disagrees with occurrences')
    for row in occurrences:
        sid = row.get('sample_id')
        pos = row.get('character_position')
        if sid not in sentence_map or isinstance(pos, bool) or (not isinstance(pos, int)) or (pos < 0):
            raise ValueError('occurrence sentence alignment is invalid')
        if row.get('occurrence_id') != f'{sid}:{pos}':
            raise ValueError('occurrence ID does not match sentence position')
        text = sentence_map[sid]['text']
        if pos >= len(text) or text[pos] != row.get('character'):
            raise ValueError(f"occurrence character alignment mismatch: {row.get('occurrence_id')}")
        parsed = parse_components(validate_canonical_label(row.get('pinyin')))
        if tuple((row.get(k) for k in ('initial', 'final', 'tone'))) != (parsed.initial, parsed.final, parsed.tone):
            raise ValueError(f"occurrence components disagree with Pinyin: {row.get('occurrence_id')}")
        positions_by_sentence[sid].append(pos)
    if any((sorted(pos) != list(range(len(sentence_map[sid]['text']))) for sid, pos in positions_by_sentence.items())):
        raise ValueError('sentence occurrence positions do not fully cover sentence text')
    occurrence_syllable = {oid: row['syllable_id'] for row in syllables for oid in row['occurrence_ids']}
    anchor_support = _anchor_q_support(syllables)
    pairs = {'dimsim-natural': _readl(root / 'pairs/dimsim-natural.jsonl'), 'dimsim-balanced': _readl(root / 'pairs/dimsim-balanced.jsonl')}
    for mode, rows in pairs.items():
        if any((r.get('mode') != ('natural' if mode.endswith('natural') else 'balanced') for r in rows)):
            raise ValueError(f'{mode} mode labels are invalid')
        draw_ids = [r.get('draw_id') for r in rows]
        config_mode = 'dimsim_' + mode.removeprefix('dimsim-')
        if len(draw_ids) != len(set(draw_ids)) or len(rows) != config['sampling'][config_mode]['count']:
            raise ValueError(f'{mode} draw count or IDs are invalid')
        if any((type(r.get('draw_index')) is not int or r.get('draw_index') != i for i, r in enumerate(rows))):
            raise ValueError(f'{mode} draw order is invalid')
        if mode.endswith('natural') and len({r.get('pair_id') for r in rows}) != len(rows):
            raise ValueError('natural DIMSIM pairs must be unique')
        for row in rows:
            if type(row.get('schema_version')) is not int or row.get('schema_version') != PAIR_SCHEMA_VERSION or type(row.get('sampler_version')) is not int or (row.get('sampler_version') != DIMSIM_SAMPLER_VERSION):
                raise ValueError(f'{mode} unsupported row or sampler version')
            a, b = (row.get('left_id'), row.get('right_id'))
            if a not in occurrence_map or b not in occurrence_map or a == b or (a >= b):
                raise ValueError(f'{mode} pair has invalid endpoints')
            left, right = (occurrence_map[a], occurrence_map[b])
            if not left['initial'] or not right['initial']:
                raise ValueError(f'{mode} violates zero-initial exclusion')
            if row.get('pair_id') != pair_identity(a, b):
                raise ValueError(f'{mode} pair identity mismatch')
            if row.get('left_pinyin') != left['pinyin'] or row.get('right_pinyin') != right['pinyin']:
                raise ValueError(f'{mode} Pinyin labels disagree with endpoint metadata')
            expected_draw = f"dimsim-{('natural' if mode.endswith('natural') else 'balanced')}:v1:{row['draw_index']:06d}"
            if row.get('draw_id') != expected_draw:
                raise ValueError(f'{mode} draw ID mismatch')
            mask = ''.join((str(int(left[k] == right[k])) for k in ('initial', 'final', 'tone')))
            if row.get('overlap_mask') != mask:
                raise ValueError(f'{mode} overlap mask mismatch')
            if any((row.get('equal_' + key) is not (mask[i] == '1') for i, key in enumerate(('initial', 'final', 'tone')))):
                raise ValueError(f'{mode} equality flags mismatch')
            if row.get('left_zero_initial') is not False or row.get('right_zero_initial') is not False or row.get('no_zero_initial_diagnostic') is not True:
                raise ValueError(f'{mode} zero-initial flags mismatch')
            if mode.endswith('balanced'):
                first_id, second_id = (row.get('first_id'), row.get('second_id'))
                if {first_id, second_id} != {a, b} or first_id == second_id:
                    raise ValueError('balanced DIMSIM orientation does not match pair endpoints')
                if row.get('first_syllable_id') != occurrence_syllable.get(first_id) or row.get('second_syllable_id') != occurrence_syllable.get(second_id):
                    raise ValueError('balanced DIMSIM syllable orientation is invalid')
            value, status = (row.get('dimsim_distance'), row.get('distance_status'))
            if status == 'ok':
                if isinstance(value, bool) or not isinstance(value, (int, float)) or (not math.isfinite(value)) or (value < 0):
                    raise ValueError(f'{mode} ok distance must be finite and non-negative')
            elif value is not None or not isinstance(status, str) or (not (status == 'not_computed' or status.startswith('invalid:'))):
                raise ValueError(f'{mode} distance status/value mismatch')
    triplets = _readl(root / 'triplets/anchor-q.jsonl')
    sampling = config.get('sampling', {}).get('anchor_q', {})
    per_edge = sampling.get('draws_per_edge')
    expected_edges = [f'{c}:{l}->{h}' for c, l, h in EDGES]
    if sampling.get('edge_order') != expected_edges or len(triplets) != len(EDGES) * per_edge:
        raise ValueError('anchor-Q edge order or draw count is invalid')
    edge_counts = {edge: 0 for edge in expected_edges}
    draw_ids = set()
    for global_index, row in enumerate(triplets):
        edge = row.get('edge_id')
        expected_global_edge = expected_edges[global_index // per_edge] if per_edge else None
        if edge != expected_global_edge:
            raise ValueError('anchor-Q rows do not follow configured global edge order')
        if edge not in edge_counts or type(row.get('schema_version')) is not int or row.get('schema_version') != TRIPLET_SCHEMA_VERSION or (type(row.get('sampler_version')) is not int) or (row.get('sampler_version') != Q_SAMPLER_VERSION):
            raise ValueError('anchor-Q row schema or edge is invalid')
        ids = [row.get(k) for k in ('anchor_id', 'high_id', 'low_id')]
        if any((x not in occurrence_map for x in ids)) or len(set(ids)) != 3:
            raise ValueError('anchor-Q occurrence references must be known and distinct')
        comp, low, high = (edge.split(':')[0], edge.split(':')[1].split('->')[0], edge.split('->')[1])
        a, h, l = (occurrence_map[x] for x in ids)
        if occurrence_syllable[ids[0]] not in anchor_support[edge][0]:
            raise ValueError('anchor-Q draw uses an ineligible anchor syllable')

        def mask(x):
            return ''.join((str(int(a[k] == x[k])) for k in ('initial', 'final', 'tone')))
        if mask(h) != high or mask(l) != low:
            raise ValueError('anchor-Q gold masks disagree with edge')
        expected_triplet = _stable('triplet:', [edge, *ids])
        if row.get('triplet_id') != expected_triplet or row.get('component') != comp or row.get('low_mask') != low or (row.get('high_mask') != high):
            raise ValueError('anchor-Q triplet identity or edge metadata mismatch')
        expected_draw_id = f"anchor-q:v1:{edge}:{row.get('draw_index', -1):06d}"
        if row.get('draw_id') != expected_draw_id:
            raise ValueError('anchor-Q draw identity mismatch')
        if high == '111' and (row.get('anchor_syllable_id') != row.get('high_syllable_id') or row.get('anchor_id') == row.get('high_id')):
            raise ValueError('anchor-Q high-111 identity rule violated')
        for name, oid in (('anchor', ids[0]), ('high', ids[1]), ('low', ids[2])):
            if row.get(name + '_syllable_id') != occurrence_syllable.get(oid):
                raise ValueError('anchor-Q syllable reference mismatch')
        if type(row.get('draw_index')) is not int or row.get('draw_index') != edge_counts[edge]:
            raise ValueError('anchor-Q draw index/order mismatch')
        if row.get('draw_id') in draw_ids:
            raise ValueError('duplicate anchor-Q draw ID')
        draw_ids.add(row['draw_id'])
        edge_counts[edge] += 1
    if any((count != per_edge for count in edge_counts.values())):
        raise ValueError('anchor-Q edge coverage counts are invalid')
    coverage = _read(root / 'coverage.json')
    expected_q_coverage = _anchor_q_coverage(syllables, triplets, seed=sampling['seed'], draws_per_edge=per_edge)
    counts = manifest.get('counts', {})
    expected_counts = {'sentences': len(sentences), 'occurrences': len(occurrences), 'syllables': len(syllables), 'dimsim_natural': len(pairs['dimsim-natural']), 'dimsim_balanced': len(pairs['dimsim-balanced']), 'anchor_q': len(triplets)}
    expected_edge_counts = {edge: per_edge for edge in expected_edges}
    if not _typed_equal(counts, expected_counts) or coverage.get('sentence_count') != len(sentences) or coverage.get('occurrence_count') != len(occurrences) or (coverage.get('anchor_q') != expected_q_coverage) or (coverage.get('anchor_q', {}).get('edges') is None) or (set(coverage.get('anchor_q', {}).get('edges', {})) != set(expected_edges)) or any((coverage['anchor_q']['edges'][edge].get('draw_count') != n for edge, n in expected_edge_counts.items())):
        raise ValueError('schema-3 coverage or manifest counts mismatch')
    expected_coverage = {'sentence_count': len(sentences), 'occurrence_count': len(occurrences), 'syllable_count': len(syllables), 'zero_initial_occurrences_excluded_from_dimsim': sum((not r['initial'] for r in occurrences)), 'dimsim_natural': {'draw_count': len(pairs['dimsim-natural']), 'unique_pairs': len({r['pair_id'] for r in pairs['dimsim-natural']}), 'unique_occurrences': len({r[k] for r in pairs['dimsim-natural'] for k in ('left_id', 'right_id')}), 'unique_syllables': len({occurrence_syllable[x] for r in pairs['dimsim-natural'] for x in (r['left_id'], r['right_id'])})}, 'dimsim_balanced': {'draw_count': len(pairs['dimsim-balanced']), 'unique_pairs': len({r['pair_id'] for r in pairs['dimsim-balanced']}), 'unique_occurrences': len({r[k] for r in pairs['dimsim-balanced'] for k in ('left_id', 'right_id')}), 'unique_syllables': len({r[k] for r in pairs['dimsim-balanced'] for k in ('first_syllable_id', 'second_syllable_id')}), 'same_pronunciation_draws': sum((r['overlap_mask'] == '111' for r in pairs['dimsim-balanced']))}, 'anchor_q': expected_q_coverage}
    if not _typed_equal(coverage, expected_coverage):
        raise ValueError('schema-3 coverage diagnostics disagree with saved samples')
    return {'root': root, 'manifest': manifest, 'config': config, 'coverage': coverage, 'sentences': sentences, 'occurrences': occurrences, 'syllables': syllables, 'pairs': pairs, 'triplets': triplets}

def _validate_id(value):
    if not isinstance(value, str) or not value or '/' in value or ('\\' in value) or ('..' in value):
        raise ValueError('invalid schema-3 evaluation_id')
    return value

def _validate_config(config, manifest_metric):
    if type(config.get('schema_version')) is not int or config['schema_version'] != SCHEMA_VERSION:
        raise ValueError('schema-3 config version must be the supported integer version')
    expected_population = {'unit': 'contextual_occurrence', 'zero_initial_excluded_from_dimsim': True, 'syllable_identity': 'canonical_toned_pinyin', 'aggregates': False}
    population = config.get('population')
    if not isinstance(population, dict) or set(population) != set(expected_population) or type(population.get('zero_initial_excluded_from_dimsim')) is not bool or (type(population.get('aggregates')) is not bool) or (not _typed_equal(population, expected_population)):
        raise ValueError('schema-3 population contract mismatch')
    sampling = config.get('sampling')
    if not isinstance(sampling, dict) or set(sampling) != {'dimsim_natural', 'dimsim_balanced', 'anchor_q'}:
        raise ValueError('schema-3 sampling contract is invalid')
    expected_keys = {'dimsim_natural': {'count', 'without_replacement', 'rng', 'seed', 'sampler_version', 'pair_schema_version'}, 'dimsim_balanced': {'count', 'with_replacement', 'rng', 'seed', 'choice_order', 'sampler_version', 'pair_schema_version'}, 'anchor_q': {'draws_per_edge', 'edge_order', 'with_replacement', 'rng', 'seed', 'sampler_version', 'triplet_schema_version'}}
    specs = (('dimsim_natural', 'random.Random', False, DIMSIM_SAMPLER_VERSION, PAIR_SCHEMA_VERSION), ('dimsim_balanced', 'numpy.random.PCG64', True, DIMSIM_SAMPLER_VERSION, PAIR_SCHEMA_VERSION), ('anchor_q', 'numpy.random.PCG64', True, Q_SAMPLER_VERSION, TRIPLET_SCHEMA_VERSION))
    for name, rng, replacement, sampler_version, schema_version in specs:
        item = sampling[name]
        if set(item) != expected_keys[name]:
            raise ValueError(f'schema-3 {name} config fields are invalid')
        count_key = 'draws_per_edge' if name == 'anchor_q' else 'count'
        count = item.get(count_key)
        seed = item.get('seed')
        if isinstance(count, bool) or not isinstance(count, int) or count <= 0 or isinstance(seed, bool) or (not isinstance(seed, int)) or (seed < 0) or (item.get('rng') != rng) or (type(item.get('sampler_version')) is not int) or (item['sampler_version'] != sampler_version) or (type(item.get('pair_schema_version' if name != 'anchor_q' else 'triplet_schema_version')) is not int) or (item['pair_schema_version' if name != 'anchor_q' else 'triplet_schema_version'] != schema_version) or (name != 'dimsim_natural' and item.get('with_replacement') is not replacement):
            raise ValueError(f'schema-3 {name} sampling contract is invalid')
    if sampling['dimsim_natural'].get('without_replacement') is not True:
        raise ValueError('natural DIMSIM must sample without replacement')
    if sampling['dimsim_balanced'].get('choice_order') != ['first_syllable', 'second_syllable', 'first_occurrence', 'second_occurrence']:
        raise ValueError('balanced DIMSIM choice order mismatch')
    score = config.get('scoring', {}).get('anchor_q', {})
    if score != {'cosine': 'stable_float64_scale_before_norm', 'tie_tolerance': 1e-12}:
        raise ValueError('anchor-Q scoring contract mismatch')
    if config.get('scoring', {}).get('dimsim') != manifest_metric:
        raise ValueError('DIMSIM scoring provenance mismatch')


## Embedded `evaluation/bundle.py`


In [9]:
"""Create and validate immutable, model-independent evaluation bundles.

The on-disk format intentionally uses JSON/JSONL so the preparation stage has
no scientific-runtime dependency. Embeddings are produced by later stages.
"""
import hashlib
import json
import random
import re
import shutil
from collections import defaultdict
from collections.abc import Callable, Iterable
from pathlib import Path
from typing import Any
BUNDLE_SCHEMA_VERSION = 2
PAIRING_POLICY = 'nonzero_nonzero'
_EVALUATION_ID_PATTERN = re.compile('^[A-Za-z0-9](?:[A-Za-z0-9._-]*[A-Za-z0-9])?$')
_WINDOWS_RESERVED_NAME_PATTERN = re.compile('^(?:CON|PRN|AUX|NUL|COM[1-9]|LPT[1-9])(?:\\..*)?$', re.IGNORECASE)
DistanceFunction = Callable[[str, str], float]

def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def _write_json(path: Path, value: Any) -> None:
    path.write_text(json.dumps(value, ensure_ascii=False, sort_keys=True, indent=2) + '\n', encoding='utf-8')

def _write_jsonl(path: Path, rows: Iterable[dict[str, Any]]) -> None:
    path.write_text(''.join((json.dumps(row, ensure_ascii=False, sort_keys=True) + '\n' for row in rows)), encoding='utf-8')

def _read_json(path: Path) -> Any:
    return json.loads(path.read_text(encoding='utf-8'))

def _read_jsonl(path: Path) -> list[dict[str, Any]]:
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line]

def _safe_id(value: str, field: str) -> str:
    if not isinstance(value, str) or not value:
        raise ValueError(f'{field} must be a non-empty string')
    return value

def _validate_evaluation_id(value: Any) -> str:
    if not isinstance(value, str) or _EVALUATION_ID_PATTERN.fullmatch(value) is None or '..' in value or ('/' in value) or ('\\' in value) or (':' in value) or value.endswith(('.', ' ')) or Path(value).is_absolute() or (_WINDOWS_RESERVED_NAME_PATTERN.fullmatch(value) is not None):
        raise ValueError("evaluation_id must be a safe single-directory name using ASCII letters, digits, '.', '_', or '-'")
    return value

def _occurrence_id(sample_id: str, position: int) -> str:
    return f'{sample_id}:{position}'

def _group_id(kind: str, values: tuple[str, ...]) -> str:
    payload = json.dumps([kind, *values], ensure_ascii=False, separators=(',', ':'))
    return f"{kind}:{hashlib.sha256(payload.encode('utf-8')).hexdigest()}"

def _load_source_experiment(root: Path, split: str, sentence_limit: int | None, sentence_seed: int) -> tuple[list[dict], dict, dict, dict]:
    files = {name: root / name for name in ('manifest.json', 'samples.jsonl', 'splits.json', 'vocab.json', 'config.json')}
    missing = [str(path) for path in files.values() if not path.is_file()]
    if missing:
        raise FileNotFoundError(missing)
    manifest = _read_json(files['manifest.json'])
    if manifest.get('pinyin_contract') != PINYIN_CONTRACT_VERSION or manifest.get('parser_provenance') != PARSER_PROVENANCE:
        raise ValueError('unsupported source experiment Pinyin contract')
    artifacts = manifest.get('artifacts', {})
    for name in ('samples.jsonl', 'splits.json', 'vocab.json', 'config.json'):
        expected = artifacts.get(name)
        if not isinstance(expected, str) or _sha256(files[name]) != expected:
            raise ValueError(f'source experiment artifact checksum mismatch: {name}')
    splits, vocab = (_read_json(files['splits.json']), _read_json(files['vocab.json']))
    if split not in splits or not isinstance(splits[split], list) or (not splits[split]):
        raise ValueError(f'unknown experiment split: {split}')
    if sentence_limit is not None and (isinstance(sentence_limit, bool) or not isinstance(sentence_limit, int) or sentence_limit <= 0):
        raise ValueError('sentence_limit must be positive or None')
    if any((not isinstance(values, list) for values in splits.values())):
        raise ValueError('source experiment splits must contain ID lists')
    all_split_ids = [sample_id for values in splits.values() for sample_id in values]
    if any((not isinstance(value, str) or not value for value in all_split_ids)) or len(all_split_ids) != len(set(all_split_ids)):
        raise ValueError('source experiment splits are not disjoint unique ID lists')
    selected_ids = sorted(splits[split])
    if sentence_limit is not None:
        if sentence_limit > len(selected_ids):
            raise ValueError(f'sentence_limit {sentence_limit} exceeds split size {len(selected_ids)}')
        selected_ids = sorted(random.Random(sentence_seed).sample(selected_ids, sentence_limit))
    sample_rows = _read_jsonl(files['samples.jsonl'])
    samples = {row['sample_id']: row for row in sample_rows}
    if len(samples) != len(sample_rows):
        raise ValueError('duplicate sample IDs in source experiment')
    try:
        selected = [samples[sample_id] for sample_id in selected_ids]
    except KeyError as error:
        raise ValueError(f'split references missing sample: {error.args[0]}') from error
    hashes = {name: _sha256(files[name]) for name in ('manifest.json', 'samples.jsonl', 'splits.json', 'vocab.json', 'config.json')}
    return (selected, vocab, manifest, hashes)

def _build_occurrences(samples: list[dict], vocab: dict) -> list[dict]:
    acoustic_tokens = vocab.get('acoustic_tokens')
    labels = vocab.get('labels', {})
    if not isinstance(acoustic_tokens, list) or any((not isinstance(x, str) for x in acoustic_tokens)):
        raise ValueError('invalid acoustic token vocabulary')
    component_names = [labels.get(name) for name in ('initials', 'finals', 'tones')]
    if any((not isinstance(values, list) or any((not isinstance(value, str) for value in values)) for values in component_names)):
        raise ValueError('invalid component label vocabulary')
    rows = []
    for sample in sorted(samples, key=lambda row: row['sample_id']):
        sample_id, text = (_safe_id(sample.get('sample_id'), 'sample_id'), sample.get('text'))
        acoustic_ids, components = (sample.get('acoustic_ids'), sample.get('component_labels'))
        if not isinstance(text, str) or not len(text) == len(acoustic_ids or []) == len(components or []):
            raise ValueError(f'source sample shape mismatch: {sample_id}')
        for position, character in enumerate(text):
            acoustic_id = acoustic_ids[position]
            component_ids = components[position]
            if not isinstance(acoustic_id, int) or acoustic_id < 0 or acoustic_id >= len(acoustic_tokens):
                raise ValueError(f'invalid acoustic ID in sample {sample_id}')
            if not isinstance(component_ids, list) or len(component_ids) != 3:
                raise ValueError(f'invalid component labels in sample {sample_id}')
            if any((isinstance(value, bool) or not isinstance(value, int) or value < 0 or (value >= len(component_names[i])) for i, value in enumerate(component_ids))):
                raise ValueError(f'component ID outside vocabulary in sample {sample_id}')
            try:
                initial, final, tone = (component_names[i][value] for i, value in enumerate(component_ids))
            except (IndexError, TypeError) as error:
                raise ValueError(f'component ID outside vocabulary in sample {sample_id}') from error
            pinyin = acoustic_tokens[acoustic_id]
            parsed = parse_components(validate_canonical_label(pinyin))
            if (initial, final, tone) != (parsed.initial, parsed.final, parsed.tone):
                raise ValueError(f'component labels disagree with acoustic label in sample {sample_id}')
            rows.append({'occurrence_id': _occurrence_id(sample_id, position), 'sample_id': sample_id, 'character_position': position, 'character': character, 'pinyin': pinyin, 'initial': initial, 'final': final, 'tone': tone})
    return sorted(rows, key=lambda row: row['occurrence_id'])

def _build_groups(occurrences: list[dict]) -> tuple[list[dict], dict[str, dict]]:
    members: dict[str, list[str]] = defaultdict(list)
    metadata: dict[str, dict] = {}
    for row in occurrences:
        definitions = (('character_pinyin', (row['character'], row['pinyin'])), ('syllable', (row['pinyin'],)))
        for kind, keys in definitions:
            group_id = _group_id(kind, keys)
            members[group_id].append(row['occurrence_id'])
            metadata[group_id] = {'group_id': group_id, 'group_kind': kind, 'keys': list(keys), 'pinyin': row['pinyin'], 'initial': row['initial'], 'final': row['final'], 'tone': row['tone']}
    groups = []
    for group_id in sorted(members):
        groups.append({**metadata[group_id], 'occurrence_ids': sorted(members[group_id]), 'occurrence_count': len(members[group_id])})
    return (groups, {row['group_id']: row for row in groups})

def _pair_rows(ids: list[str], metadata: dict[str, dict], *, sample_count: int | None, seed: int, max_all_pairs: int, distance_fn: DistanceFunction | None) -> list[dict]:
    rows = []
    for left_id, right_id in make_pair_plan(ids, sample_count=sample_count, seed=seed, max_all_pairs=max_all_pairs):
        left, right = (metadata[left_id], metadata[right_id])
        status, distance = ('not_computed', None)
        if distance_fn is not None:
            try:
                distance = float(distance_fn(left['pinyin'], right['pinyin']))
                if distance != distance or distance in (float('inf'), float('-inf')):
                    raise ValueError('distance is not finite')
                status = 'ok'
            except Exception as error:
                status = f'invalid:{type(error).__name__}'
                distance = None
        equal_initial = left['initial'] == right['initial']
        equal_final = left['final'] == right['final']
        equal_tone = left['tone'] == right['tone']
        pair_id = hashlib.sha256(json.dumps([left_id, right_id], ensure_ascii=False, separators=(',', ':')).encode()).hexdigest()
        rows.append({'pair_id': pair_id, 'left_id': left_id, 'right_id': right_id, 'left_pinyin': left['pinyin'], 'right_pinyin': right['pinyin'], 'dimsim_distance': distance, 'distance_status': status, 'equal_initial': equal_initial, 'equal_final': equal_final, 'equal_tone': equal_tone, 'overlap_mask': f'{int(equal_initial)}{int(equal_final)}{int(equal_tone)}', 'left_zero_initial': left['initial'] == '', 'right_zero_initial': right['initial'] == '', 'no_zero_initial_diagnostic': left['initial'] != '' and right['initial'] != ''})
    return rows

def prepare_evaluation_bundle(evaluation_id: str, *, experiment_path: str | Path, output_root: str | Path, split: str='test', sentence_limit: int | None=None, contextual_pair_count: int=12000, sentence_seed: int=42, sampling_seed: int=42, character_pair_count: int=12000, syllable_pair_count: int | None=None, max_all_pairs: int=1000000, distance_fn: DistanceFunction | None=None, metric_provenance: dict[str, Any] | None=None, code_provenance: dict[str, str] | None=None, protocol_config: dict[str, Any] | None=None) -> Path:
    """Create an immutable evaluation bundle and return its path."""
    if protocol_config is not None:
        return prepare_contextual_bundle(evaluation_id, experiment_path=experiment_path, output_root=output_root, split=split, sentence_limit=sentence_limit, sentence_seed=sentence_seed, distance_fn=distance_fn, metric_provenance=metric_provenance, code_provenance=code_provenance, protocol_config=protocol_config)
    evaluation_id = _validate_evaluation_id(evaluation_id)
    numeric = {'contextual_pair_count': contextual_pair_count, 'sentence_seed': sentence_seed, 'sampling_seed': sampling_seed, 'character_pair_count': character_pair_count, 'max_all_pairs': max_all_pairs}
    if any((isinstance(value, bool) or not isinstance(value, int) for value in numeric.values())):
        raise TypeError('pair counts, caps, and seeds must be integers')
    if contextual_pair_count < 0 or character_pair_count < 0 or max_all_pairs <= 0:
        raise ValueError('pair counts must be non-negative and max_all_pairs positive')
    if syllable_pair_count is not None and (isinstance(syllable_pair_count, bool) or not isinstance(syllable_pair_count, int) or syllable_pair_count < 0):
        raise ValueError('syllable_pair_count must be a non-negative integer or None')
    if distance_fn is None or not callable(distance_fn):
        raise ValueError('an explicit validated distance provider is required')
    if not isinstance(metric_provenance, dict) or not isinstance(metric_provenance.get('metric_name'), str):
        raise ValueError('metric_provenance with an explicit metric_name is required')
    metric_name = metric_provenance['metric_name']
    if metric_name == 'dimsim':
        if not isinstance(distance_fn, DimsimDistance) or metric_provenance != distance_fn.provenance:
            raise ValueError('DIMSIM bundles require the validated pinned DimsimDistance provider and its provenance')
    elif metric_name != 'synthetic':
        raise ValueError('only the pinned DIMSIM provider or an explicitly labelled synthetic test provider is supported')
    if not isinstance(code_provenance, dict) or not code_provenance or any((not isinstance(key, str) or not isinstance(value, str) or len(value) != 64 or any((character not in '0123456789abcdef' for character in value)) for key, value in code_provenance.items())):
        raise ValueError('explicit non-empty preparation code_provenance hashes are required')
    experiment_root, root = (Path(experiment_path).resolve(), Path(output_root).resolve())
    target = (root / evaluation_id).resolve()
    if target.parent != root:
        raise ValueError('evaluation path escapes output_root')
    if target == experiment_root or experiment_root in target.parents:
        raise ValueError('evaluation output must not be inside the source experiment')
    if target.exists():
        raise FileExistsError(f'evaluation bundle already exists: {target}')
    samples, vocab, source_manifest, source_hashes = _load_source_experiment(experiment_root, split, sentence_limit, sentence_seed)
    occurrences = _build_occurrences(samples, vocab)
    groups, group_by_id = _build_groups(occurrences)
    occurrence_by_id = {row['occurrence_id']: row for row in occurrences}
    eligible_ids = {'contextual': sorted((row['occurrence_id'] for row in occurrences if isinstance(row['initial'], str) and row['initial'])), 'character_pinyin': sorted((row['group_id'] for row in groups if row['group_kind'] == 'character_pinyin' and isinstance(row['initial'], str) and row['initial'])), 'syllable': sorted((row['group_id'] for row in groups if row['group_kind'] == 'syllable' and isinstance(row['initial'], str) and row['initial']))}
    contextual = _pair_rows(eligible_ids['contextual'], occurrence_by_id, sample_count=contextual_pair_count, seed=sampling_seed, max_all_pairs=max_all_pairs, distance_fn=distance_fn)
    aggregate_plans = {}
    pair_counts = {'character_pinyin': character_pair_count, 'syllable': syllable_pair_count}
    for offset, kind in enumerate(('character_pinyin', 'syllable'), start=1):
        aggregate_plans[kind] = _pair_rows(eligible_ids[kind], group_by_id, sample_count=pair_counts[kind], seed=sampling_seed + offset, max_all_pairs=max_all_pairs, distance_fn=distance_fn)
    target.mkdir(parents=True)
    try:
        (target / 'pairs').mkdir()
        selected_sentences = [{'sample_id': row['sample_id'], 'text': row['text']} for row in sorted(samples, key=lambda x: x['sample_id'])]
        _write_jsonl(target / 'sentences.jsonl', selected_sentences)
        _write_jsonl(target / 'occurrences.jsonl', occurrences)
        _write_jsonl(target / 'groups.jsonl', groups)
        _write_jsonl(target / 'pairs/contextual.jsonl', contextual)
        for kind, rows in aggregate_plans.items():
            _write_jsonl(target / f'pairs/{kind}.jsonl', rows)
        object_counts = {'contextual': len(occurrences), 'character_pinyin': sum((row['group_kind'] == 'character_pinyin' for row in groups)), 'syllable': sum((row['group_kind'] == 'syllable' for row in groups))}
        all_plans = {'contextual': contextual, **aggregate_plans}
        coverage = {'sentences': len(samples), 'occurrences': len(occurrences), 'groups': {kind: object_counts[kind] for kind in ('character_pinyin', 'syllable')}, 'levels': {}}
        for level, rows in all_plans.items():
            statuses: dict[str, int] = {}
            for row in rows:
                statuses[row['distance_status']] = statuses.get(row['distance_status'], 0) + 1
            original_count = object_counts[level]
            eligible_count = len(eligible_ids[level])
            original_pairs, eligible_pairs = (pair_count(original_count), pair_count(eligible_count))
            coverage['levels'][level] = {'objects': {'original': original_count, 'eligible': eligible_count, 'excluded': original_count - eligible_count, 'excluded_by_reason': {'zero_initial': original_count - eligible_count}}, 'unordered_pair_population': {'original': original_pairs, 'eligible': eligible_pairs, 'excluded': original_pairs - eligible_pairs, 'excluded_by_reason': {'zero_initial': original_pairs - eligible_pairs}}, 'selection_count': len(rows), 'distance_status_counts': dict(sorted(statuses.items()))}
        _write_json(target / 'coverage.json', coverage)
        config = {'schema_version': BUNDLE_SCHEMA_VERSION, 'evaluation_id': evaluation_id, 'split': split, 'sentence_limit': sentence_limit, 'contextual_pair_count': contextual_pair_count, 'sentence_seed': sentence_seed, 'sampling_seed': sampling_seed, 'character_pair_count': character_pair_count, 'syllable_pair_count': syllable_pair_count, 'max_all_pairs': max_all_pairs, 'pairing_policy': PAIRING_POLICY}
        _write_json(target / 'config.json', config)
        artifacts = {str(path.relative_to(target)).replace('\\', '/'): _sha256(path) for path in target.rglob('*') if path.is_file()}
        manifest = {'schema_version': BUNDLE_SCHEMA_VERSION, 'evaluation_id': evaluation_id, 'pinyin_contract': PINYIN_CONTRACT_VERSION, 'parser_provenance': PARSER_PROVENANCE, 'pairing_policy': PAIRING_POLICY, 'source_experiment': {'path': str(experiment_root), 'experiment_id': source_manifest.get('experiment_id'), 'hashes': source_hashes}, 'metric': metric_provenance, 'code_provenance': code_provenance, 'counts': {'sentences': len(samples), 'occurrences': len(occurrences), 'character_pinyin_groups': sum((row['group_kind'] == 'character_pinyin' for row in groups)), 'syllable_groups': sum((row['group_kind'] == 'syllable' for row in groups)), 'pairs': {'contextual': len(contextual), **{kind: len(rows) for kind, rows in aggregate_plans.items()}}}, 'artifacts': artifacts}
        _write_json(target / 'manifest.json', manifest)
    except Exception:
        shutil.rmtree(target)
        raise
    return target

def load_evaluation_bundle(path: str | Path) -> dict[str, Any]:
    """Load a bundle after verifying every recorded artifact checksum."""
    root = Path(path).resolve()
    manifest = _read_json(root / 'manifest.json')
    if manifest.get('schema_version') == 3:
        return load_contextual_bundle(root, manifest)
    if manifest.get('schema_version') != BUNDLE_SCHEMA_VERSION:
        raise ValueError('unsupported evaluation bundle schema')
    if manifest.get('pinyin_contract') != PINYIN_CONTRACT_VERSION or manifest.get('parser_provenance') != PARSER_PROVENANCE:
        raise ValueError('unsupported evaluation Pinyin contract')
    config = _read_json(root / 'config.json')
    if config.get('schema_version') != BUNDLE_SCHEMA_VERSION:
        raise ValueError('unsupported evaluation config schema')
    if manifest.get('pairing_policy') != PAIRING_POLICY or config.get('pairing_policy') != PAIRING_POLICY:
        raise ValueError('unsupported evaluation pairing policy')
    manifest_evaluation_id = _validate_evaluation_id(manifest.get('evaluation_id'))
    config_evaluation_id = _validate_evaluation_id(config.get('evaluation_id'))
    if manifest_evaluation_id != config_evaluation_id:
        raise ValueError('invalid or inconsistent evaluation_id')
    required = {'config.json', 'coverage.json', 'sentences.jsonl', 'occurrences.jsonl', 'groups.jsonl', 'pairs/contextual.jsonl', 'pairs/character_pinyin.jsonl', 'pairs/syllable.jsonl'}
    artifacts = manifest.get('artifacts', {})
    if not required.issubset(artifacts):
        raise ValueError('evaluation manifest is missing required artifacts')
    for relative, expected in artifacts.items():
        artifact = (root / relative).resolve()
        if root not in artifact.parents or not artifact.is_file() or _sha256(artifact) != expected:
            raise ValueError(f'evaluation artifact checksum mismatch: {relative}')
    coverage = _read_json(root / 'coverage.json')
    occurrences, groups = (_read_jsonl(root / 'occurrences.jsonl'), _read_jsonl(root / 'groups.jsonl'))
    occurrence_ids = [row.get('occurrence_id') for row in occurrences]
    group_ids = [row.get('group_id') for row in groups]
    if len(occurrence_ids) != len(set(occurrence_ids)) or any((not isinstance(value, str) for value in occurrence_ids)):
        raise ValueError('evaluation occurrence IDs are invalid or duplicated')
    if len(group_ids) != len(set(group_ids)) or any((not isinstance(value, str) for value in group_ids)):
        raise ValueError('evaluation group IDs are invalid or duplicated')
    if any((not isinstance(row.get('initial'), str) for row in occurrences + groups)):
        raise ValueError('evaluation initial metadata must be strings')
    for row in occurrences + groups:
        label = validate_canonical_label(row.get('pinyin'))
        parsed = parse_components(label)
        if tuple((row.get(key) for key in ('initial', 'final', 'tone'))) != (parsed.initial, parsed.final, parsed.tone):
            raise ValueError('stored evaluation components disagree with canonical Pinyin label')
    try:
        _, expected_groups = _build_groups(occurrences)
    except (KeyError, TypeError) as error:
        raise ValueError('invalid occurrence group metadata') from error
    if {row['group_id']: row for row in groups} != expected_groups:
        raise ValueError('stored evaluation group metadata is inconsistent with occurrences')
    pairs = {kind: _read_jsonl(root / f'pairs/{kind}.jsonl') for kind in ('contextual', 'character_pinyin', 'syllable')}
    occurrence_id_set = set(occurrence_ids)
    valid_ids = {'contextual': occurrence_id_set, 'character_pinyin': {row['group_id'] for row in groups if row.get('group_kind') == 'character_pinyin'}, 'syllable': {row['group_id'] for row in groups if row.get('group_kind') == 'syllable'}}
    metadata = {'contextual': {row['occurrence_id']: row for row in occurrences}, 'character_pinyin': {row['group_id']: row for row in groups if row.get('group_kind') == 'character_pinyin'}, 'syllable': {row['group_id']: row for row in groups if row.get('group_kind') == 'syllable'}}
    for group in groups:
        members = group.get('occurrence_ids')
        if not isinstance(members, list) or not members or members != sorted(members) or (len(members) != len(set(members))) or (group.get('occurrence_count') != len(members)) or any((member not in occurrence_id_set for member in members)):
            raise ValueError(f"invalid group membership: {group.get('group_id')}")
    for kind, rows in pairs.items():
        pair_ids = [row.get('pair_id') for row in rows]
        if len(pair_ids) != len(set(pair_ids)) or any((not isinstance(value, str) for value in pair_ids)):
            raise ValueError(f'{kind} pair IDs are invalid or duplicated')
        if any((row.get('left_id') not in valid_ids[kind] or row.get('right_id') not in valid_ids[kind] for row in rows)):
            raise ValueError(f'{kind} pair references an unknown object')
        endpoints = [(row['left_id'], row['right_id']) for row in rows]
        if len(endpoints) != len(set(endpoints)) or any((left >= right for left, right in endpoints)):
            raise ValueError(f'{kind} pairs are duplicated or non-canonical')
        for row in rows:
            expected = hashlib.sha256(json.dumps([row['left_id'], row['right_id']], ensure_ascii=False, separators=(',', ':')).encode()).hexdigest()
            if row['pair_id'] != expected:
                raise ValueError(f'{kind} pair ID does not match endpoints')
            left = metadata[kind][row['left_id']]
            right = metadata[kind][row['right_id']]
            left_zero = metadata[kind][row['left_id']].get('initial') == ''
            right_zero = metadata[kind][row['right_id']].get('initial') == ''
            if left_zero or right_zero:
                raise ValueError(f'{kind} pair violates {PAIRING_POLICY} policy')
            if row.get('left_zero_initial') is not False or row.get('right_zero_initial') is not False or row.get('no_zero_initial_diagnostic') is not True:
                raise ValueError(f'{kind} pair has inconsistent zero-initial flags')
            if row.get('left_pinyin') != left['pinyin'] or row.get('right_pinyin') != right['pinyin']:
                raise ValueError(f'{kind} pair Pinyin labels disagree with endpoints')
            equal = [left[key] == right[key] for key in ('initial', 'final', 'tone')]
            if any((row.get('equal_' + key) is not value for key, value in zip(('initial', 'final', 'tone'), equal))) or row.get('overlap_mask') != ''.join((str(int(value)) for value in equal)):
                raise ValueError(f'{kind} pair component metadata disagrees with endpoints')
        original_count = len(valid_ids[kind])
        eligible_count = sum((isinstance(metadata[kind][object_id].get('initial'), str) and bool(metadata[kind][object_id]['initial']) for object_id in valid_ids[kind]))
        original_pairs, eligible_pairs = (pair_count(original_count), pair_count(eligible_count))
        statuses: dict[str, int] = {}
        for row in rows:
            status = row.get('distance_status')
            statuses[status] = statuses.get(status, 0) + 1
        expected_coverage = {'objects': {'original': original_count, 'eligible': eligible_count, 'excluded': original_count - eligible_count, 'excluded_by_reason': {'zero_initial': original_count - eligible_count}}, 'unordered_pair_population': {'original': original_pairs, 'eligible': eligible_pairs, 'excluded': original_pairs - eligible_pairs, 'excluded_by_reason': {'zero_initial': original_pairs - eligible_pairs}}, 'selection_count': len(rows), 'distance_status_counts': dict(sorted(statuses.items()))}
        if coverage.get('levels', {}).get(kind) != expected_coverage:
            raise ValueError(f'{kind} coverage metadata is inconsistent')
    expected_counts = {'sentences': len(_read_jsonl(root / 'sentences.jsonl')), 'occurrences': len(occurrences), 'character_pinyin_groups': len(valid_ids['character_pinyin']), 'syllable_groups': len(valid_ids['syllable']), 'pairs': {kind: len(rows) for kind, rows in pairs.items()}}
    if manifest.get('counts') != expected_counts:
        raise ValueError('evaluation manifest counts are inconsistent')
    return {'root': root, 'manifest': manifest, 'config': config, 'coverage': coverage, 'sentences': _read_jsonl(root / 'sentences.jsonl'), 'occurrences': occurrences, 'groups': groups, 'pairs': pairs}


In [10]:
from pathlib import Path

def find_project_root(start=None):
    current = Path.cwd() if start is None else Path(start)
    current = current.resolve()
    for candidate in (current, *current.parents):
        if (candidate / "src" / "phonetics").is_dir():
            return candidate
        child = candidate / "project"
        if (child / "src" / "phonetics").is_dir():
            return child
    raise FileNotFoundError("Could not find project root from current path")

# Local defaults. On Kaggle, set both custom paths, for example:
# CUSTOM_EXPERIMENT_PATH = Path("/kaggle/input/mandarin-experiment-1-0-0/1.0.0")
# CUSTOM_OUTPUT_ROOT = Path("/kaggle/working/evaluations")
CUSTOM_EXPERIMENT_PATH = None
CUSTOM_OUTPUT_ROOT = None
EXPERIMENT_ID = "duyu-chinese-lessons-zh-cn-v1"
EVALUATION_ID = "duyu-chinese-lessons-zh-cn-v1"
if (CUSTOM_EXPERIMENT_PATH is None) != (CUSTOM_OUTPUT_ROOT is None):
    raise ValueError("Set both custom preparation paths or neither")
if CUSTOM_EXPERIMENT_PATH is None:
    PROJECT_ROOT = find_project_root()
    EXPERIMENT_PATH = PROJECT_ROOT / "datasets" / "experiments" / EXPERIMENT_ID
    OUTPUT_ROOT = PROJECT_ROOT / "datasets" / "evaluations"
else:
    EXPERIMENT_PATH = Path(CUSTOM_EXPERIMENT_PATH)
    OUTPUT_ROOT = Path(CUSTOM_OUTPUT_ROOT)
SPLIT = "test"
SENTENCE_LIMIT = None
SENTENCE_SEED = 42
DIMSIM_NATURAL_PAIR_COUNT = 12_000
DIMSIM_NATURAL_SEED = 42
DIMSIM_BALANCED_DRAW_COUNT = 12_000
DIMSIM_BALANCED_SEED = 20_261_026
ANCHOR_Q_DRAWS_PER_EDGE = 1_000
ANCHOR_Q_SEED = 20_261_026



In [11]:
metric = DimsimDistance()
PROTOCOL_CONFIG = {
    "dimsim_natural_count": DIMSIM_NATURAL_PAIR_COUNT,
    "dimsim_natural_seed": DIMSIM_NATURAL_SEED,
    "dimsim_balanced_count": DIMSIM_BALANCED_DRAW_COUNT,
    "dimsim_balanced_seed": DIMSIM_BALANCED_SEED,
    "anchor_q_draws_per_edge": ANCHOR_Q_DRAWS_PER_EDGE,
    "anchor_q_seed": ANCHOR_Q_SEED,
}
bundle_path = prepare_evaluation_bundle(
    EVALUATION_ID, experiment_path=EXPERIMENT_PATH, output_root=OUTPUT_ROOT, split=SPLIT,
    sentence_limit=SENTENCE_LIMIT, sentence_seed=SENTENCE_SEED,
    distance_fn=metric, metric_provenance=metric.provenance,
    code_provenance=BUILD_METADATA["source_sha256"], protocol_config=PROTOCOL_CONFIG,
)
print(bundle_path)


D:\projects\Mandarin-Phonetics-Modeling\project\datasets\evaluations\duyu-chinese-lessons-zh-cn-v1
